In [5]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import pvlib


In [6]:
study_areas = Path(r'/Users/peter.jackson/projects/tassie-trees-research-plan/data/vector/study_areas.geojson')

study_areas_gdf = gpd.read_file(study_areas)

dates = {
    "Davey River": "2023-02-17",
    "Wilson River": "2025-04-30",
    "Harman River": "2025-04-30",
    "Stanley River": "2025-04-30",
}

# Get centroids and transform to WGS84
centroids_wgs84 = study_areas_gdf.geometry.centroid.to_crs(epsg=4326)
study_areas_gdf['lon'] = centroids_wgs84.x
study_areas_gdf['lat'] = centroids_wgs84.y

# Map dates and create timezone-aware datetime objects for noon local time in Tasmania
study_areas_gdf['date'] = study_areas_gdf['name'].map(dates)
study_areas_gdf['datetime'] = pd.to_datetime(study_areas_gdf['date'] + ' 12:00:00').dt.tz_localize('Australia/Hobart')

# Calculate sun elevation and azimuth
def get_sun_pos(row):
    time_idx = pd.DatetimeIndex([row['datetime']])
    sol_pos = pvlib.solarposition.get_solarposition(time_idx, row['lat'], row['lon'])
    return pd.Series({
        'sun_elevation': sol_pos['elevation'].iloc[0],
        'sun_azimuth': sol_pos['azimuth'].iloc[0]
    })

study_areas_gdf[['sun_elevation', 'sun_azimuth']] = study_areas_gdf.apply(get_sun_pos, axis=1)

In [7]:
study_areas_gdf

,fid,name,geometry,lon,lat,date,datetime,sun_elevation,sun_azimuth
0,1,Harman River,"POLYGON ((363113.206 5382249.651, 364885.802 5...",145.348140,-41.654046,2025-04-30,2025-04-30 12:00:00+10:00,33.415347,4.585803
1,2,Stanley River,"POLYGON ((359894.085 5389888.15, 358153.496 53...",145.303782,-41.669975,2025-04-30,2025-04-30 12:00:00+10:00,33.396805,4.636232
2,3,Wilson River,"POLYGON ((366301.504 5389790.802, 363782.562 5...",145.378624,-41.665860,2025-04-30,2025-04-30 12:00:00+10:00,33.405384,4.549943
3,13,Davey River,"POLYGON ((416604.414 5226265.188, 422126.709 5...",145.989858,-43.040690,2023-02-17,2023-02-17 12:00:00+11:00,53.476405,38.966010
